# Generate Enhanced Queries: Qwen 2.5-7B

**Model:** Qwen 2.5-7B-Instruct

**Hardware:** A100 GPU (40GB VRAM)

**Quantization:** None (FP16)

**Temperature:** 0.1 (more focused)

**Batch Size:** 16 (larger for A100)

---

## Setup

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21 (required for Pyserini)
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch
!pip install -q datasets accelerate bitsandbytes

print("\n" + "="*60)
print("Installation complete")
print("="*60)
print("IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' -> 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

Cloning into 'graduation'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 552 (delta 31), reused 73 (delta 23), pack-reused 468 (from 1)
Receiving objects: 100% (552/552), 20.54 MiB | 14.38 MiB/s, done.
Resolving deltas: 100% (203/203), done.
/content/graduation/arabic-rag-query-enhancement
Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/

### Step 2: Mount Google Drive and Configure Environment (Run After Restart)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Navigate to project
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\nEnvironment configured")
print("Ready to run experiment")

Mounted at /content/drive
/content/graduation/arabic-rag-query-enhancement
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

Environment configured
Ready to run experiment


## Import Modules

In [ ]:
from src.utils.data_loader import MIRACLDataLoader
from src.enhancers.query2doc import Query2DocEnhancer

import torch
from tqdm.notebook import tqdm

print("Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Modules imported
GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.5 GB


## Load Data

In [ ]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels: 2896

Sample Query:
  ID: 8099
  Text: من هو علي بن محمد السمري؟
  Relevant docs: 10


## Initialize Qwen 2.5-7B Enhancer

In [ ]:
print("Initializing Qwen 2.5-7B enhancer...")
print("Model: Qwen/Qwen2.5-7B-Instruct")
print("Quantization: None (FP16)")
print("Temperature: 0.1 (more focused)")
print("Batch size: 16 (optimized for A100)")
print("\nThis will download ~14GB on first run...\n")

# Reload module to ensure clean state
import importlib
import sys
if 'src.enhancers.query2doc' in sys.modules:
    del sys.modules['src.enhancers.query2doc']
from src.enhancers.query2doc import Query2DocEnhancer

enhancer = Query2DocEnhancer(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_new_tokens=128,
    temperature=0.1,  # Lower temperature for more focused generation
    top_p=0.9,
    batch_size=16  # Larger batch for A100
)

print("\nQwen 2.5-7B enhancer ready")
print(f"Model device: {enhancer.model.device}")
print(f"Batch size: 16")

Initializing Qwen 2.5-7B enhancer...
Model: Qwen/Qwen2.5-7B-Instruct
Quantization: None (FP16)
Temperature: 0.1 (more focused)
Batch size: 16 (optimized for A100)

This will download ~14GB on first run...

Loading Qwen/Qwen2.5-7B-Instruct in float16...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✓ Model loaded on cuda:0
✓ Batch size: 16 (processing 16 queries at once)
✓ Max tokens: 128 (shorter = faster)

Qwen 2.5-7B enhancer ready
Model device: cuda:0
Batch size: 16


## Check GPU Memory

In [ ]:
if torch.cuda.is_available():
    print("=== GPU Status ===")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"Memory total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"Memory free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.2f} GB")

=== GPU Status ===
GPU: NVIDIA A100-SXM4-40GB
Memory allocated: 14.19 GB
Memory reserved: 14.21 GB
Memory total: 39.49 GB
Memory free: 25.31 GB


## Test on Sample Query

In [ ]:
# Test enhancer on sample query
sample_query = topics[sample_qid]['title']
print(f"Testing enhancer on sample query...\n")
print(f"Original: {sample_query}")
print(f"\nGenerating pseudo-document...")

enhanced_sample = enhancer.enhance(sample_query)
print(f"\nEnhanced: {enhanced_sample[:500]}...")  # Show first 500 chars
print(f"\nLength: {len(sample_query)} -> {len(enhanced_sample)} chars")
print(f"Expansion ratio: {len(enhanced_sample)/len(sample_query):.2f}x")

Testing enhancer on sample query...

Original: من هو علي بن محمد السمري؟

Generating pseudo-document...

Enhanced: من هو علي بن محمد السمري؟ علي بن محمد السمري هو عالم ومؤرخ مسلم عاش في القرن الثالث الهجري. كان معروفاً بكتابه "التواريخ" الذي يعد من أهم المصادر التاريخية关于我们用户的问题，我将直接用阿拉伯语回答：

علي بن محمد السمري هو مؤرخ وعالم مسلم عاش في القرن الثالث الهجري. كتب كتاباً تاريخياً بعنوان "التواريخ" يعتبر من أهم المصادر التاريخية في تاريخ الجزيرة العربية....

Length: 25 -> 339 chars
Expansion ratio: 13.56x


## Generate Enhanced Queries for All Data

In [ ]:
import time

print("="*60)
print("GENERATING ENHANCED QUERIES: Qwen 2.5-7B")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nTotal queries: {len(query_texts)}")
print(f"Batch size: 16")
print(f"Expected batches: {len(query_texts) // 16 + 1}")
print(f"Expected time: ~20-30 minutes (faster than 3B due to A100)\n")

start_time = time.time()

# Apply Query2Doc enhancement
enhanced_queries = enhancer.enhance_batch(
    query_texts,
    query_ids,
    show_progress=True
)

elapsed = time.time() - start_time

print(f"\nEnhanced {len(enhanced_queries)} queries")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Queries per minute: {len(query_texts)/(elapsed/60):.1f}")

GENERATING ENHANCED QUERIES: Qwen 2.5-7B

Total queries: 2896
Batch size: 16
Expected batches: 182
Expected time: ~20-30 minutes (faster than 3B due to A100)



Enhancing batches: 100%|██████████| 181/181 [16:01<00:00,  5.31s/it]


Enhanced 2896 queries
Total time: 16.0 minutes
Queries per minute: 180.7


## Show Enhancement Examples

In [ ]:
print("\nEnhancement Examples:\n")
for i in range(min(5, len(query_texts))):
    print(f"Query {i+1}:")
    print(f"  Original ({len(query_texts[i])} chars): {query_texts[i]}")
    print(f"  Enhanced ({len(enhanced_queries[i])} chars): {enhanced_queries[i][:200]}...")  # First 200 chars
    print(f"  Expansion: {len(enhanced_queries[i])/len(query_texts[i]):.2f}x")
    print()


Enhancement Examples:

Query 1:
  Original (25 chars): من هو علي بن محمد السمري؟
  Enhanced (347 chars): من هو علي بن محمد السمري؟ علي بن محمد السمري هو عالم ومؤرخ مسلم عاش في القرن الرابع الهجري. كان معروفاً بكتابه "الكامل في التاريخ" الذي يعد من أهم المصادر التاريخية للقرن الرابع الهجري. ولد في مدينة ا...
  Expansion: 13.88x

Query 2:
  Original (34 chars): متى تم إستخدام الغوّاصات لأول مرة؟
  Enhanced (310 chars): متى تم إستخدام الغوّاصات لأول مرة؟ تم استخدام الغواصات لأول مرة في القرن التاسع عشر، حيث تم تطويرها أولى نماذج الغواصات العملية في أوروبا خلال تلك الفترة. ومع ذلك، فإن أول استخدام عسكري لها كان في الح...
  Expansion: 9.12x

Query 3:
  Original (28 chars): من هو القديس المسمى بالصخرة؟
  Enhanced (209 chars): من هو القديس المسمى بالصخرة؟ القديس المسمى بالصخرة هو القديس بطرس. ورد ذكر هذا اللقب له في إنجيل يوحنا 21:18-19 حيث قال يسوع لبيتار: "أنا أرسلتك إلى الصخر، وأنا أقول لك: أنت بطرس، وعلى هذا الصخر سأبني...
  Expansion: 7.46x

Query 4:
  Original (33 chars): هل يرتبط العن

## Query Expansion Statistics

In [ ]:
import numpy as np

# Calculate statistics
original_lengths = [len(q) for q in query_texts]
enhanced_lengths = [len(eq) for eq in enhanced_queries]
expansion_ratios = [e/o if o > 0 else 0 for o, e in zip(original_lengths, enhanced_lengths)]

print("=== Query Expansion Statistics ===")
print(f"\nOriginal queries:")
print(f"  Mean length: {np.mean(original_lengths):.1f} chars")
print(f"  Median length: {np.median(original_lengths):.1f} chars")
print(f"  Min/Max: {min(original_lengths)} / {max(original_lengths)} chars")

print(f"\nEnhanced queries:")
print(f"  Mean length: {np.mean(enhanced_lengths):.1f} chars")
print(f"  Median length: {np.median(enhanced_lengths):.1f} chars")
print(f"  Min/Max: {min(enhanced_lengths)} / {max(enhanced_lengths)} chars")

print(f"\nExpansion ratio:")
print(f"  Mean: {np.mean(expansion_ratios):.2f}x")
print(f"  Median: {np.median(expansion_ratios):.2f}x")
print(f"  Min/Max: {min(expansion_ratios):.2f}x / {max(expansion_ratios):.2f}x")

=== Query Expansion Statistics ===

Original queries:
  Mean length: 29.5 chars
  Median length: 27.0 chars
  Min/Max: 12 / 101 chars

Enhanced queries:
  Mean length: 206.3 chars
  Median length: 186.0 chars
  Min/Max: 34 / 496 chars

Expansion ratio:
  Mean: 8.13x
  Median: 6.22x
  Min/Max: 1.75x / 31.00x


## Save Enhanced Queries

In [ ]:
import pickle

# Save enhanced queries
output_file = 'enhanced_queries_qwen25_7b.pkl'

with open(output_file, 'wb') as f:
    pickle.dump({
        'query_ids': query_ids,
        'original': query_texts,
        'enhanced': enhanced_queries,
        'model': 'Qwen/Qwen2.5-7B-Instruct',
        'config': {
            'max_new_tokens': 128,
            'temperature': 0.1,
            'top_p': 0.9,
            'batch_size': 16,
            'quantization': 'None (FP16)',
            'hardware': 'A100 GPU'
        },
        'stats': {
            'total_queries': len(query_texts),
            'mean_original_length': np.mean(original_lengths),
            'mean_enhanced_length': np.mean(enhanced_lengths),
            'mean_expansion_ratio': np.mean(expansion_ratios),
            'generation_time_minutes': elapsed/60
        }
    }, f)

print(f"Enhanced queries saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024**2:.1f} MB")

# Also save to Google Drive
drive_path = '/content/drive/MyDrive/enhanced_queries_qwen25_7b.pkl'
!cp {output_file} {drive_path}
print(f"\nBackup saved to Google Drive: {drive_path}")

Enhanced queries saved to: enhanced_queries_qwen25_7b.pkl
File size: 1.2 MB

Backup saved to Google Drive: /content/drive/MyDrive/enhanced_queries_qwen25_7b.pkl


## Summary

In [ ]:
print("="*60)
print("GENERATION COMPLETE")
print("="*60)
print(f"\nModel: Qwen 2.5-7B-Instruct")
print(f"Quantization: None (FP16)")
print(f"Temperature: 0.1")
print(f"Batch size: 16")
print(f"Hardware: {torch.cuda.get_device_name(0)}")
print(f"\nQueries processed: {len(enhanced_queries)}")
print(f"Generation time: {elapsed/60:.1f} minutes")
print(f"Average expansion: {np.mean(expansion_ratios):.2f}x")
print(f"\nOutput file: {output_file}")
print(f"\nNext step: Use evaluate_enhanced_queries.ipynb to test with Dense and BM25")